#Common Crawl: Framing Drift Analysis Pipeline

**Purpose**: Scalable pipeline for measuring framing drift in online news over time.

**Workflow**:
1. Loop through quarterly windows (3-month periods) across a date range
2. Apply reservoir sampling to cap stories per window
3. Deterministically align each story to appropriate Common Crawl indices by publish date
4. Retrieve full text from Common Crawl WARC archives
5. Generate model-ready outputs:
   - Index CSV: audit trail with metadata
   - Payload JSONL: full text + lead_400w + body_800w slices

**Data Flow**: Common Crawl (full text) -> Framing classifier

**NOTE**: Claude Code was used extensively in this notebook to enhance print statements and try/catch logic while troubleshooting. While this proved invaluable to overcoming several hurdles, insurmountable challenges with URL formation, index sampling, and article availability persisted. This pipeline was ultimately abandoned, as the time commitment had already taken critical days away from model development and dataset analysis. Included for transparency.

In [ ]:
# Cell 0: Install dependencies
%pip install trafilatura warcio requests langdetect python-dateutil

In [ ]:
# Cell 1: Imports
import os, io, json, gzip, requests, csv, random, hashlib
from urllib.parse import urlparse, urlsplit, urlunsplit
from datetime import date, timedelta, datetime
from dateutil.relativedelta import relativedelta
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

from warcio.archiveiterator import ArchiveIterator
import trafilatura
from langdetect import detect, LangDetectException

# === Progress Logging Setup ===
LOG_FILE = Path("./pipeline_progress.log")

def log_progress(message, also_print=True):
    """
    Write progress message to log file with timestamp.
    Also prints to stdout so it appears in notebook.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_line = f"[{timestamp}] {message}\n"
    
    # Write to log file (can be tailed in real-time)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(log_line)
    
    # Also print to stdout for notebook
    if also_print:
        print(f"[{timestamp}] {message}", flush=True)

# Clear/create log file
with open(LOG_FILE, "w") as f:
    f.write(f"=== Pipeline started at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ===\n")

log_progress(f"Log file created: {LOG_FILE.absolute()}")
log_progress(f"Monitor progress with: tail -f {LOG_FILE.absolute()}")

In [ ]:
# Cell 2: Configuration

# === Target Outlet ===
OUTLET_DOMAIN = "propublica.org"  # Single outlet per run
OUTLET_NAME = "ProPublica"  # Human-readable name for output

# === Date Range & Time Windows ===
START_DATE = date.fromisoformat("2019-01-01")
END_DATE = date.fromisoformat("2023-12-31")  # Inclusive
WINDOW_MONTHS = 3  # Quarterly windows (3 months each)

# === TEST MODE ===
TEST_MODE = True  # Set to True to process only Q1 2019 for testing
if TEST_MODE:
    END_DATE = date.fromisoformat("2019-03-31")  # Just Q1 2019 for testing
    print("TEST MODE ENABLED - Processing only Q1 2019")

# === Sampling ===
MAX_ARTICLES_PER_WINDOW = 100  # Target number of usable articles per window
OVERSAMPLE_MULTIPLIER = 20     # Oversampling to account for retrieval failures

# === Parallel Processing ===
MAX_WORKERS = 8  # Number of parallel WARC fetches (adjust based on network/CPU)

# === Common Crawl ===
MIN_TEXT_LENGTH = 300  # Minimum extracted text length to consider success

# === Output ===
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

INDEX_CSV = OUTPUT_DIR / f"{OUTLET_DOMAIN.replace('.', '_')}_index.csv"
PAYLOAD_JSONL = OUTPUT_DIR / f"{OUTLET_DOMAIN.replace('.', '_')}_payload.jsonl"

print(f"Configuration loaded:")
print(f"  Outlet: {OUTLET_NAME} ({OUTLET_DOMAIN})")
print(f"  Date range: {START_DATE} to {END_DATE}")
print(f"  Window size: {WINDOW_MONTHS} months (quarterly)")
print(f"  Max articles per window: {MAX_ARTICLES_PER_WINDOW} (with {OVERSAMPLE_MULTIPLIER}x oversampling)")
print(f"  Parallel workers: {MAX_WORKERS} concurrent WARC fetches")
print(f"  Output: {INDEX_CSV.name}, {PAYLOAD_JSONL.name}")

In [ ]:
# Cell 2.5: Outlet Pattern Configuration
# ============================================
# INSTRUCTIONS:
# 1. Open OUTLET_PATTERN_CONFIGS.txt
# 2. Find the configuration block for your target outlet
# 3. Copy the entire block (between the comment bars)
# 4. Paste it here, replacing the example below
# 5. Run this cell
# ============================================

# Example: ProPublica Configuration
# Replace this entire block with your outlet's config from OUTLET_PATTERN_CONFIGS.txt

URL_PATTERNS = [
    "/article/",
    "/atpropublica/"
]

EXCLUSION_PATTERNS = [
    "/series/",
    "/video/",
    "/podcasts/"
]

# ============================================
# Do not modify below this line
# ============================================

print(f"Outlet pattern configuration loaded")
print(f"  Domain: {OUTLET_DOMAIN}")
print(f"  Name: {OUTLET_NAME}")
print(f"  URL patterns to INCLUDE: {URL_PATTERNS}")
print(f"  URL patterns to EXCLUDE: {EXCLUSION_PATTERNS}")
print(f"\n  Articles must match at least one INCLUDE pattern AND not match any EXCLUDE pattern")

In [ ]:
# Cell 3: Helper Functions - URL Variants & CDX Lookup

def url_variants(url):
    """
    Generate URL variants for Common Crawl matching:
    - Original URL
    - Stripped (no query/fragment)
    - Trailing slash variants
    """
    parts = list(urlsplit(url))
    variants = [url]
    
    # Stripped version (no query/fragment)
    parts[3] = ""  # query
    parts[4] = ""  # fragment
    stripped = urlunsplit(parts)
    variants.append(stripped)
    
    # Toggle trailing slash on path
    if parts[2].endswith("/"):
        parts2 = parts.copy()
        parts2[2] = parts2[2].rstrip("/")
        variants.append(urlunsplit(parts2))
    else:
        parts2 = parts.copy()
        parts2[2] = parts2[2] + "/"
        variants.append(urlunsplit(parts2))
    
    # Dedupe while preserving order
    seen, out = set(), []
    for v in variants:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out


def cdx_lookup_first(url, index_id, use_prefix=False, timeout=30):
    """
    Query Common Crawl CDX index for a URL.
    Returns first matching CDX record or None.
    """
    params = {"url": url, "output": "json"}
    if use_prefix:
        params["matchType"] = "prefix"
    
    try:
        r = requests.get(
            f"https://index.commoncrawl.org/{index_id}-index",
            params=params,
            timeout=timeout
        )
        if r.status_code != 200 or not r.text.strip():
            return None
        
        # Parse first JSON line
        return json.loads(r.text.splitlines()[0])
    except (requests.RequestException, json.JSONDecodeError, IndexError):
        return None


def fetch_warc_html_bytes(cdx_row, timeout=60):
    """
    Fetch WARC record from Common Crawl using byte-range request.
    Returns raw HTML bytes from HTTP response body.
    """
    warc_path = cdx_row["filename"]
    offset = int(cdx_row["offset"])
    length = int(cdx_row["length"])
    end = offset + length - 1
    
    try:
        r = requests.get(
            f"https://data.commoncrawl.org/{warc_path}",
            headers={"Range": f"bytes={offset}-{end}"},
            timeout=timeout
        )
        r.raise_for_status()
        
        # Parse WARC and extract HTTP response body
        for rec in ArchiveIterator(io.BytesIO(r.content)):
            if rec.rec_type == "response":
                payload = rec.content_stream().read()
                # Split HTTP headers from body (double CRLF separator)
                sep = payload.find(b"\r\n\r\n")
                return payload[sep+4:] if sep != -1 else payload
        return None
    except requests.RequestException:
        return None


def extract_text_and_metadata(html_bytes, url_hint):
    """
    Extract article text AND metadata (title, date, author) from HTML using Trafilatura.
    
    UPDATED: Now extracts publish date from HTML <meta> tags to validate article dates.
    
    Returns:
        dict with 'text', 'title', 'date', 'author' (all may be None)
        OR None if extraction fails
    """
    if not html_bytes:
        return None
    
    try:
        html_str = html_bytes.decode("utf-8", errors="ignore")
        
        # Extract with metadata using JSON output format
        result = trafilatura.extract(
            html_str,
            url=url_hint,
            include_comments=False,
            include_tables=False,
            output_format='json',  # Returns JSON with metadata!
            with_metadata=True
        )
        
        if result:
            data = json.loads(result)
            return {
                'text': data.get('text'),
                'title': data.get('title'),
                'date': data.get('date'),      # ISO format YYYY-MM-DD from <meta> tags
                'author': data.get('author')
            }
        
        return None
    except Exception:
        return None


print("Helper functions loaded: url_variants, cdx_lookup_first, fetch_warc_html_bytes, extract_text_and_metadata")

In [ ]:
# Cell 4: Common Crawl Index Selection & Domain Queries

def get_cc_index_metadata(max_retries=3, backoff_seconds=2):
    """
    Fetch metadata for all available Common Crawl indices.
    Returns list of dicts with id, name.
    """
    for attempt in range(max_retries):
        try:
            r = requests.get("https://index.commoncrawl.org/collinfo.json", timeout=30)
            r.raise_for_status()
            
            indices = []
            for item in r.json():
                if "id" in item:
                    indices.append({
                        "id": item["id"],
                        "name": item.get("name", item["id"]),
                        "_raw": item
                    })
            
            return indices
            
        except (requests.ConnectionError, requests.Timeout) as e:
            if attempt < max_retries - 1:
                wait_time = backoff_seconds * (2 ** attempt)
                print(f"Warning: Failed to fetch CC indices (attempt {attempt + 1}/{max_retries}): {type(e).__name__}")
                print(f"  Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Failed to fetch CC indices after {max_retries} attempts")
                raise


def select_indices_for_date_range(start_date, end_date, all_indices):
    """
    Select CC indices that overlap with the given date range.
    
    Strategy: CC indices are named CC-MAIN-YYYY-WW (year-week)
    We select indices from the window period + 1 month buffer for late publications.
    """
    start_year = start_date.year
    end_year = end_date.year
    start_month = start_date.month
    end_month = end_date.month
    
    # Add 1 month buffer after end_date for late publications/crawls
    buffer_end_month = end_month + 1
    buffer_end_year = end_year
    if buffer_end_month > 12:
        buffer_end_month = 1
        buffer_end_year += 1
    
    # Improved week calculation using ISO week numbers
    import datetime as dt
    start_week = max(1, dt.date(start_year, start_month, 1).isocalendar()[1] - 1)
    
    # For buffer end, use the actual ISO week
    if buffer_end_month == 1:
        buffer_end_week_date = dt.date(buffer_end_year, 1, 15)
    else:
        buffer_end_week_date = dt.date(buffer_end_year if buffer_end_month > 1 else buffer_end_year, buffer_end_month, 1)
    end_week = min(53, buffer_end_week_date.isocalendar()[1] + 1)
    
    relevant_indices = []
    
    for idx in all_indices:
        idx_id = idx["id"]
        try:
            # Parse CC-MAIN-YYYY-WW format
            parts = idx_id.split("-")
            if len(parts) >= 3 and parts[0] == "CC" and parts[1] == "MAIN":
                idx_year = int(parts[2])
                
                # Check if year matches our range
                if idx_year < start_year or idx_year > buffer_end_year:
                    continue
                
                # For the target year(s), check week number if available
                if len(parts) >= 4:
                    try:
                        idx_week = int(parts[3])
                        
                        # If start and end are in same year
                        if start_year == buffer_end_year == idx_year:
                            if start_week <= idx_week <= end_week:
                                relevant_indices.append(idx)
                        # If spanning years
                        elif idx_year == start_year and start_week <= idx_week:
                            relevant_indices.append(idx)
                        elif idx_year == buffer_end_year and idx_week <= end_week:
                            relevant_indices.append(idx)
                    except ValueError:
                        # If week parsing fails, include it to be safe
                        if start_year <= idx_year <= buffer_end_year:
                            relevant_indices.append(idx)
                else:
                    # No week number, just use year
                    if start_year <= idx_year <= buffer_end_year:
                        relevant_indices.append(idx)
        except (ValueError, IndexError):
            pass
    
    # Sort by ID (chronological)
    relevant_indices.sort(key=lambda x: x["id"])
    
    return relevant_indices


def query_domain_in_index(domain, index_id, timeout=120):
    """
    Query Common Crawl CDX index for ALL URLs from a domain.
    Returns list of CDX records with url, timestamp, filename, offset, length.
    """
    url = f"https://index.commoncrawl.org/{index_id}-index"
    params = {
        "url": f"*.{domain}",
        "output": "json",
        "fl": "url,timestamp,filename,offset,length,digest"
    }
    
    try:
        r = requests.get(url, params=params, timeout=120)
        if r.status_code != 200:
            print(f"  Warning: CDX query failed for {domain} in {index_id}: HTTP {r.status_code}")
            return []
        
        if not r.text.strip():
            return []
        
        # Parse all JSON lines
        records = []
        for line in r.text.strip().split('\n'):
            if line:
                try:
                    record = json.loads(line)
                    records.append(record)
                except json.JSONDecodeError:
                    continue
        
        return records
        
    except requests.Timeout:
        print(f"  Warning: CDX query timeout for {domain} in {index_id}")
        return []
    except requests.RequestException as e:
        print(f"  Warning: CDX query failed for {domain} in {index_id}: {type(e).__name__}")
        return []


# Cache CC indices metadata (fetched once per notebook run)
print("Fetching Common Crawl index metadata...")
CC_INDICES_METADATA = get_cc_index_metadata()
print(f"Loaded {len(CC_INDICES_METADATA)} Common Crawl indices")
print(f"  Most recent: {CC_INDICES_METADATA[0]['id'] if CC_INDICES_METADATA else 'None'}")
print(f"  Oldest: {CC_INDICES_METADATA[-1]['id'] if CC_INDICES_METADATA else 'None'}")
print(f"\nStrategy: Query domain (*.{OUTLET_DOMAIN}), use tight index selection + metadata date validation")

In [ ]:
# Cell 5: Text Slicing for Classifier

def split_into_words(text):
    """
    Simple word tokenization by whitespace.
    More sophisticated tokenization (nltk) could be added if needed.
    """
    return text.split()


def extract_lead_400w(text):
    """
    Extract first ~400 words (lead of article).
    """
    words = split_into_words(text)
    return " ".join(words[:400])


def extract_body_800w(text):
    """
    Extract next ~800 words after lead (body of article).
    """
    words = split_into_words(text)
    return " ".join(words[400:1200])  # Words 401-1200


def detect_language(text):
    """
    Detect language using langdetect.
    Returns ISO 639-1 code (e.g., 'en') or None on failure.
    """
    try:
        # Use first 1000 chars for detection (faster, sufficient)
        return detect(text[:1000])
    except LangDetectException:
        return None


print("Text processing functions loaded: extract_lead_400w, extract_body_800w, detect_language")

In [ ]:
# Cell 6: Helper Functions for Filtering and Sampling

def filter_by_date_range(cdx_records, start_date, end_date):
    """
    Filter CDX records by timestamp to match date range.
    CDX timestamp format: YYYYMMDDHHMMSS
    """
    filtered = []
    
    start_ts = int(start_date.strftime("%Y%m%d") + "000000")
    end_ts = int(end_date.strftime("%Y%m%d") + "235959")
    
    for record in cdx_records:
        try:
            timestamp = int(record.get("timestamp", "0"))
            if start_ts <= timestamp <= end_ts:
                filtered.append(record)
        except (ValueError, AttributeError):
            continue
    
    return filtered


def filter_articles_only(cdx_records):
    """
    Filter to only article URLs using configured URL_PATTERNS.
    Excludes URLs matching EXCLUSION_PATTERNS.
    
    UPDATED (Phase 2): Now uses URL_PATTERNS and EXCLUSION_PATTERNS from Cell 2.5.
    This allows flexible configuration for different outlets with different URL schemas.
    """
    filtered = []
    
    for record in cdx_records:
        url = record.get('url', '')
        
        # Check if URL matches any inclusion pattern
        matches_include = any(pattern in url for pattern in URL_PATTERNS)
        
        # Check if URL matches any exclusion pattern
        matches_exclude = any(pattern in url for pattern in EXCLUSION_PATTERNS)
        
        # Include if matches include pattern AND doesn't match exclude pattern
        if matches_include and not matches_exclude:
            filtered.append(record)
    
    return filtered


def deduplicate_by_url(cdx_records):
    """
    Remove duplicate URLs, keeping only the first occurrence.
    """
    seen_urls = set()
    unique = []
    
    for record in cdx_records:
        url = record.get('url', '')
        if url and url not in seen_urls:
            seen_urls.add(url)
            unique.append(record)
    
    return unique


def sample_articles(cdx_records, max_count):
    """
    Sample articles if we have more than max_count.
    Uses random sampling for diversity.
    """
    if len(cdx_records) <= max_count:
        return cdx_records
    
    return random.sample(cdx_records, max_count)


def parse_cdx_timestamp(timestamp_str):
    """
    Convert CDX timestamp (YYYYMMDDHHMMSS) to date object.
    """
    try:
        year = int(timestamp_str[0:4])
        month = int(timestamp_str[4:6])
        day = int(timestamp_str[6:8])
        return date(year, month, day)
    except:
        return None


print("Helper functions loaded: filter_by_date_range, filter_articles_only, deduplicate_by_url, sample_articles")

In [ ]:
# Cell 7: Main Processing Functions

def generate_time_windows(start_date, end_date, window_months=3):
    """
    Generate (start, end) date pairs for time windows of specified length.
    """
    windows = []
    current = start_date.replace(day=1)
    
    while current <= end_date:
        next_window = current + relativedelta(months=window_months)
        window_end = next_window - timedelta(days=1)
        
        if window_end > end_date:
            window_end = end_date
        
        windows.append((current, window_end))
        current = next_window
    
    return windows


def process_cdx_record(cdx_record, outlet_domain, outlet_name, window_start, window_end, min_text_length=300):
    """
    Process a single CDX record: fetch WARC, extract text, validate date.
    
    UPDATED: Now validates article publish date from HTML metadata.
    Articles with dates outside the window are rejected to prevent temporal contamination.
    
    Returns dict with status, text, metadata, and date validation info.
    """
    url = cdx_record.get("url", "")
    cc_timestamp = cdx_record.get("timestamp", "")
    
    # Fetch WARC content
    html_bytes = fetch_warc_html_bytes(cdx_record)
    extracted = extract_text_and_metadata(html_bytes, url)
    
    if not extracted or not extracted.get('text'):
        return {
            "status": "failure",
            "failure_reason": "extraction_failed",
            "url": url,
            "publish_date": "",
            "date_source": "",
            "title": "",
            "cc_timestamp": cc_timestamp,
            "warc_filename": cdx_record.get("filename", ""),
            "warc_offset": cdx_record.get("offset", ""),
            "warc_length": cdx_record.get("length", ""),
            "sha1": cdx_record.get("digest", ""),
            "language": None,
            "text_full": None,
            "text_length": 0
        }
    
    text = extracted['text']
    title = extracted.get('title', '')
    metadata_date = extracted.get('date')  # ISO string from HTML <meta> tags
    
    # === DATE VALIDATION ===
    # Priority: metadata date > CDX timestamp
    final_date = None
    date_source = None
    
    if metadata_date:
        try:
            # Trafilatura returns YYYY-MM-DD format
            final_date = date.fromisoformat(metadata_date)
            date_source = "metadata"
        except (ValueError, TypeError):
            pass
    
    if not final_date:
        # Fallback to CDX timestamp
        final_date = parse_cdx_timestamp(cc_timestamp)
        date_source = "cdx_timestamp"
    
    # FIX #3: Relax date validation - allow articles published up to 7 days before window start
    # This accounts for edge cases where articles are published at month boundaries
    relaxed_start = window_start - timedelta(days=7)
    
    # Check if date is within relaxed window
    if final_date and (final_date < relaxed_start or final_date > window_end):
        return {
            "status": "failure",
            "failure_reason": "date_out_of_range",
            "url": url,
            "publish_date": final_date.isoformat(),
            "date_source": date_source,
            "title": title,
            "cc_timestamp": cc_timestamp,
            "warc_filename": cdx_record.get("filename", ""),
            "warc_offset": cdx_record.get("offset", ""),
            "warc_length": cdx_record.get("length", ""),
            "sha1": cdx_record.get("digest", ""),
            "language": None,
            "text_full": None,
            "text_length": 0
        }
    
    # === VALIDATION PASSED ===
    if len(text.strip()) >= min_text_length:
        lang = detect_language(text)
        
        return {
            "status": "success",
            "failure_reason": "",
            "url": url,
            "publish_date": final_date.isoformat() if final_date else "",
            "date_source": date_source or "",
            "title": title,
            "cc_timestamp": cc_timestamp,
            "warc_filename": cdx_record.get("filename", ""),
            "warc_offset": cdx_record.get("offset", ""),
            "warc_length": cdx_record.get("length", ""),
            "sha1": cdx_record.get("digest", ""),
            "language": lang,
            "text_full": text,
            "text_length": len(text)
        }
    else:
        return {
            "status": "failure",
            "failure_reason": "text_too_short",
            "url": url,
            "publish_date": final_date.isoformat() if final_date else "",
            "date_source": date_source or "",
            "title": title,
            "cc_timestamp": cc_timestamp,
            "warc_filename": cdx_record.get("filename", ""),
            "warc_offset": cdx_record.get("offset", ""),
            "warc_length": cdx_record.get("length", ""),
            "sha1": cdx_record.get("digest", ""),
            "language": None,
            "text_full": None,
            "text_length": 0
        }


print("Main processing functions loaded")

In [ ]:
# Cell 8: Execute Pipeline

print("="*80)
print("Starting framing drift analysis pipeline")
print(f"Outlet: {OUTLET_NAME} ({OUTLET_DOMAIN})")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Window size: {WINDOW_MONTHS} months")
print(f"Parallel workers: {MAX_WORKERS}")
print("="*80)

# Track overall timing
pipeline_start_time = time.time()

log_progress("="*80)
log_progress(f"PIPELINE START: {OUTLET_NAME} ({OUTLET_DOMAIN})")
log_progress(f"Date range: {START_DATE} to {END_DATE}")
log_progress(f"Window size: {WINDOW_MONTHS} months")
log_progress(f"Strategy: {MAX_WORKERS}x parallel WARC fetching with {OVERSAMPLE_MULTIPLIER}x oversampling")
log_progress("="*80)

# Generate time windows
time_windows = generate_time_windows(START_DATE, END_DATE, WINDOW_MONTHS)
print(f"\nProcessing {len(time_windows)} time window(s) of {WINDOW_MONTHS} months each")
print(f"Parallel processing: {MAX_WORKERS} concurrent WARC fetches")
print(f"Date validation: Articles with dates outside window will be rejected\n")

log_progress(f"Total windows to process: {len(time_windows)}")
log_progress("")

# Accumulators for output
all_results = []
success_count = 0
failure_count = 0
date_rejected_count = 0

# Thread-safe lock for shared counters
results_lock = threading.Lock()

def process_article_worker(args):
    """
    Worker function for parallel processing.
    Returns (result_dict, success_flag, date_rejected_flag)
    """
    cdx_record, outlet_domain, outlet_name, window_start, window_end, min_text_length = args
    
    result = process_cdx_record(cdx_record, outlet_domain, outlet_name, window_start, window_end, min_text_length)
    
    success = (result["status"] == "success")
    date_rejected = (result.get("failure_reason") == "date_out_of_range")
    
    return (result, success, date_rejected)


# Process each time window
for window_num, (window_start, window_end) in enumerate(time_windows, 1):
    window_start_time = time.time()
    
    print(f"\n{'='*80}")
    print(f"Window {window_num}/{len(time_windows)}: {window_start.strftime('%Y-%m-%d')} to {window_end.strftime('%Y-%m-%d')}")
    print(f"{'='*80}")
    
    log_progress("")
    log_progress(f"{'='*60}")
    log_progress(f"WINDOW {window_num}/{len(time_windows)}: {window_start.strftime('%Y-%m-%d')} to {window_end.strftime('%Y-%m-%d')}")
    log_progress(f"{'='*60}")
    
    # Select CC indices for this time period
    indices_for_window = select_indices_for_date_range(window_start, window_end, CC_INDICES_METADATA)
    print(f"  Selected {len(indices_for_window)} CC indices for this period")
    log_progress(f"[INDICES] Selected {len(indices_for_window)} indices for {window_start.year}")
    
    # Query each CC index for domain URLs
    print(f"  Querying {len(indices_for_window)} CC indices for {OUTLET_DOMAIN}...")
    log_progress(f"[CDX QUERY] Querying {len(indices_for_window)} indices...")
    
    all_cdx_records = []
    for idx_num, idx in enumerate(indices_for_window, 1):
        idx_id = idx["id"]
        print(f"    [{idx_num}/{len(indices_for_window)}] Querying {idx_id}...", end="", flush=True)
        
        records = query_domain_in_index(OUTLET_DOMAIN, idx_id)
        all_cdx_records.extend(records)
        print(f" {len(records)} URLs")
        
        # Rate limiting between index queries
        if idx_num < len(indices_for_window):
            time.sleep(3)
    
    print(f"  Total URLs found: {len(all_cdx_records):,}")
    log_progress(f"[CDX QUERY] Found {len(all_cdx_records):,} total URLs")
    
    # Apply buffer consistently - use same 1-month buffer as index selection
    buffer_end_date = window_end + relativedelta(months=1)
    
    # Filter by date range (including buffer)
    filtered_by_date = filter_by_date_range(all_cdx_records, window_start, buffer_end_date)
    print(f"  URLs in date range (with buffer to {buffer_end_date.strftime('%Y-%m-%d')}): {len(filtered_by_date):,}")
    log_progress(f"[FILTER DATE] {len(filtered_by_date):,} URLs in date range (window + 1mo buffer)")
    
    # Filter to articles only
    articles_only = filter_articles_only(filtered_by_date)
    print(f"  Article URLs: {len(articles_only):,}")
    log_progress(f"[FILTER ARTICLES] {len(articles_only):,} article URLs")
    
    # Deduplicate
    unique_articles = deduplicate_by_url(articles_only)
    print(f"  Unique articles: {len(unique_articles):,}")
    log_progress(f"[DEDUPE] {len(unique_articles):,} unique articles")
    
    if not unique_articles:
        print("  Warning: No articles found for this window, skipping")
        log_progress(f"[SKIP] No articles found, moving to next window")
        continue
    
    # Sample with aggressive oversampling
    sample_size = min(len(unique_articles), int(MAX_ARTICLES_PER_WINDOW * OVERSAMPLE_MULTIPLIER))
    sampled_records = sample_articles(unique_articles, sample_size)
    print(f"  Sampled: {len(sampled_records)} articles (target: {MAX_ARTICLES_PER_WINDOW}, {OVERSAMPLE_MULTIPLIER}x oversampling)")
    log_progress(f"[SAMPLE] Selected {len(sampled_records)}/{len(unique_articles)} articles")
    
    # Process articles in parallel
    print(f"  Processing articles with {MAX_WORKERS} parallel workers...")
    log_progress(f"[WARC FETCH] Starting parallel retrieval for {len(sampled_records)} URLs...")
    
    window_success = 0
    window_failure = 0
    window_date_rejected = 0
    window_results = []
    processed = 0
    
    # Prepare worker arguments
    worker_args = [
        (cdx_record, OUTLET_DOMAIN, OUTLET_NAME, window_start, window_end, MIN_TEXT_LENGTH)
        for cdx_record in sampled_records
    ]
    
    # Process in parallel with ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all jobs
        future_to_idx = {executor.submit(process_article_worker, args): idx 
                        for idx, args in enumerate(worker_args)}
        
        # Process results as they complete
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            processed += 1
            
            try:
                result, success, date_rejected = future.result()
                
                # Add metadata
                result["outlet_domain"] = OUTLET_DOMAIN
                result["outlet_name"] = OUTLET_NAME
                result["window"] = f"{window_start.strftime('%Y-%m')}_to_{window_end.strftime('%Y-%m')}"
                result["window_start"] = window_start.isoformat()
                result["window_end"] = window_end.isoformat()
                
                window_results.append(result)
                
                if success:
                    window_success += 1
                    text_len = result.get('text_length', 0)
                    url_short = result['url'][:60]
                    print(f"    [{processed}/{len(sampled_records)}] Success: {url_short}... ({text_len} chars)")
                else:
                    window_failure += 1
                    url_short = result['url'][:60]
                    failure_reason = result.get('failure_reason', 'unknown')
                    
                    if date_rejected:
                        window_date_rejected += 1
                        pub_date = result.get('publish_date', 'unknown')
                        if processed % 10 == 0:  # Only print every 10th date mismatch to reduce noise
                            print(f"    [{processed}/{len(sampled_records)}] Failed: {url_short}... (date: {pub_date})")
                    else:
                        if processed % 10 == 0:  # Reduce noise
                            print(f"    [{processed}/{len(sampled_records)}] Failed: {url_short}... ({failure_reason})")
                
                # Log progress every 50 articles
                if processed % 50 == 0:
                    log_progress(
                        f"[PROGRESS] {processed}/{len(sampled_records)} processed | "
                        f"Success: {window_success} | Fail: {window_failure} | Date rejected: {window_date_rejected} | "
                        f"Rate: {window_success/processed*100:.0f}%"
                    )
                
                # Early stopping if we've reached target
                if window_success >= MAX_ARTICLES_PER_WINDOW:
                    print(f"    Reached target of {MAX_ARTICLES_PER_WINDOW} articles, stopping early")
                    log_progress(f"[EARLY STOP] Reached target at {processed}/{len(sampled_records)}")
                    # Cancel remaining futures
                    for f in future_to_idx.keys():
                        f.cancel()
                    break
                    
            except Exception as e:
                print(f"    [{processed}/{len(sampled_records)}] Worker error: {type(e).__name__}")
                window_failure += 1
    
    # Add window results to global results
    all_results.extend(window_results)
    success_count += window_success
    failure_count += window_failure
    date_rejected_count += window_date_rejected
    
    window_time = time.time() - window_start_time
    
    print(f"\n  Window summary: {window_success} successes, {window_failure} failures ({window_date_rejected} date mismatches)")
    if (window_success + window_failure) > 0:
        success_rate = window_success/(window_success+window_failure)*100
        print(f"  Success rate: {success_rate:.1f}%")
        print(f"  Time: {window_time/60:.1f} min ({processed} articles processed)")
    
    log_progress(
        f"[WINDOW COMPLETE] {window_success} successes, {window_failure} failures ({window_date_rejected} date mismatches) | "
        f"Success rate: {success_rate:.0f}% | "
        f"Time: {window_time/60:.1f} min"
    )
    
    # Overall progress estimate
    elapsed = time.time() - pipeline_start_time
    avg_window_time = elapsed / window_num
    est_total_time = avg_window_time * len(time_windows)
    est_remaining = est_total_time - elapsed
    
    log_progress(
        f"[OVERALL] Completed {window_num}/{len(time_windows)} windows | "
        f"Total: {success_count} successes, {failure_count} failures ({date_rejected_count} date mismatches) | "
        f"Elapsed: {elapsed/60:.1f} min | Est. remaining: {est_remaining/60:.1f} min"
    )

pipeline_time = time.time() - pipeline_start_time

print(f"\n{'='*80}")
print(f"Pipeline complete")
print(f"Total: {success_count} successes, {failure_count} failures ({date_rejected_count} date mismatches)")
print(f"Runtime: {pipeline_time/60:.1f} minutes")

log_progress("")
log_progress("="*80)
log_progress(f"PIPELINE COMPLETE")
log_progress(f"Total runtime: {pipeline_time/60:.1f} minutes")
log_progress(f"Total articles: {success_count} successes, {failure_count} failures")
log_progress(f"Date filtering: {date_rejected_count} articles rejected for date mismatch")

total_attempts = success_count + failure_count
if total_attempts > 0:
    final_rate = success_count/total_attempts*100
    date_reject_rate = date_rejected_count/total_attempts*100 if total_attempts > 0 else 0
    print(f"Overall success rate: {final_rate:.1f}%")
    print(f"Date rejection rate: {date_reject_rate:.1f}% (prevented temporal contamination)")
    print(f"Average per window: {success_count/len(time_windows):.1f} usable articles")
    log_progress(f"Overall success rate: {final_rate:.1f}%")
    log_progress(f"Date rejection rate: {date_reject_rate:.1f}%")
    log_progress(f"Average per window: {success_count/len(time_windows):.1f} usable articles")
else:
    print("No articles were processed")
    log_progress("ERROR: No articles were processed")

print(f"{'='*80}")
log_progress("="*80)

In [ ]:
# Cell 9: Generate Output Files

print("\nGenerating output files...\n")
log_progress("")
log_progress("[OUTPUT] Generating output files...")

# === Write Index CSV ===
print(f"Writing index CSV: {INDEX_CSV}")

csv_headers = [
    "outlet_name",
    "outlet_domain",
    "window",
    "window_start",
    "window_end",
    "url",
    "publish_date",
    "date_source",
    "title",
    "status",
    "failure_reason",
    "cc_timestamp",
    "warc_filename",
    "warc_offset",
    "warc_length",
    "sha1",
    "language",
    "text_length"
]

with open(INDEX_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=csv_headers, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(all_results)

print(f"  Wrote {len(all_results)} rows")
log_progress(f"[OUTPUT] Index CSV written: {INDEX_CSV} ({len(all_results)} rows)")

# === Write Payload JSONL ===
print(f"\nWriting payload JSONL: {PAYLOAD_JSONL}")

payload_count = 0
with open(PAYLOAD_JSONL, "w", encoding="utf-8") as f:
    for result in all_results:
        if result["status"] == "success" and result["text_full"]:
            text_full = result["text_full"]
            
            # Ensure all date fields are strings
            publish_date = result["publish_date"]
            if isinstance(publish_date, date):
                publish_date = publish_date.isoformat()
            
            window_start = result["window_start"]
            if isinstance(window_start, date):
                window_start = window_start.isoformat()
            
            window_end = result["window_end"]
            if isinstance(window_end, date):
                window_end = window_end.isoformat()
            
            payload = {
                "url": result["url"],
                "publish_date": publish_date,
                "date_source": result.get("date_source", ""),
                "outlet_domain": result["outlet_domain"],
                "outlet_name": result["outlet_name"],
                "window": result["window"],
                "window_start": window_start,
                "window_end": window_end,
                "title": result.get("title", ""),
                "language": result["language"],
                "text_full": text_full,
                "lead_400w": extract_lead_400w(text_full),
                "body_800w": extract_body_800w(text_full),
                "cc_timestamp": result.get("cc_timestamp", ""),
                "sha1": result["sha1"]
            }
            
            f.write(json.dumps(payload, ensure_ascii=False) + "\n")
            payload_count += 1

print(f"  Wrote {payload_count} article payloads")
log_progress(f"[OUTPUT] Payload JSONL written: {PAYLOAD_JSONL} ({payload_count} articles)")

print(f"\n{'='*80}")
print(f"Output files ready:")
print(f"  - Index CSV: {INDEX_CSV} ({len(all_results)} rows)")
print(f"  - Payload JSONL: {PAYLOAD_JSONL} ({payload_count} articles)")
print(f"\nNext step: Feed {PAYLOAD_JSONL} to framing classifier")
print(f"{'='*80}")

log_progress("")
log_progress("="*80)
log_progress("[COMPLETE] All output files generated successfully")
log_progress(f"  - Index CSV: {INDEX_CSV}")
log_progress(f"  - Payload JSONL: {PAYLOAD_JSONL}")
log_progress("="*80)
log_progress(f"Pipeline finished at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log_progress("="*80)